In [1]:
# 全局设置
import os
import warnings
warnings.filterwarnings('ignore')
import datetime as dt

import numpy as np
import pandas as pd
from IPython.display import Markdown

from QuantStudio.Tools.Visualization import qs_help

# 风险库

In [2]:
# 创建风险库
from QuantStudio.Risk.HDF5RDB import HDF5FRDB

RDB = HDF5FRDB(args={"MainDir": "../data/Risk"}).connect()

print(qs_help(RDB))

2026-03-18 22:28:09,508 | QS | WARNING : 找不到配置文件: C:\Users\hst\QuantStudioConfig\HDF5FRDBConfig.json


类型: HDF5FRDB
模块: QuantStudio.Risk.HDF5RDB
文档:
    基于 HDF5 文件的多因子风险数据库


In [3]:
# 风险库的参数集
display(Markdown(RDB.Args.info()))

* Name(名称): <class 'str'>, 默认值 HDF5FRDB
* MainDir(主目录): <class 'pathlib.Path'>, 无默认值, 存放数据的主目录

## 风险表列表

In [4]:
# 风险库中的所有风险表列表
RDB.TableNames

['demo_risk_table']

# 风险表

## 创建风险表

In [5]:
# 获取风险库中的某个风险表对象
RT = RDB.getTable("demo_risk_table")
print(qs_help(RT))

类型: HDF5FactorRiskTable
模块: QuantStudio.Risk.HDF5RDB
文档:
    基于 HDF5 文件的多因子风险表


In [6]:
# 风险表的参数集
display(Markdown(RT.Args.info()))

* Name: <class 'str'>, 默认值 风险表

## 元信息

In [7]:
# 获取风险表元信息的方法
print(qs_help(RT.getMetaData))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.getMetaData(key: Optional[str] = None) -> Union[pandas.core.series.Series, Any]
文档:
    获取风险表的元信息, 元信息由若干个键值对组成
    
    Args:
        key: 元信息键, None 表示获取所有的元信息
    
    Returns:
        如果 key 非 None 则返回该 key 对应的元信息
        如果 key=None, 则返回 Series(index=[所有的 key])


In [9]:
# 读取风险表的所有元信息
print(RT.getMetaData())

Description    这是一张示例风险表
dtype: object


In [10]:
# 读取风险表的某个元信息
print(RT.getMetaData(key="Description"))

这是一张示例风险表


## 时点序列

getDateTime(start_dt=None, end_dt=None):
* start_dt: datetime 或者 None, 起始时点
* end_dt: datetime 或者 None, 终止时点
* 返回: list(datetime)

In [11]:
# 获取时点序列的方法
print(qs_help(RT.getDateTime))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.getDateTime(start_dt: Optional[datetime.datetime] = None, end_dt: Optional[datetime.datetime] = None) -> List[datetime.datetime]
文档:
    获取时点序列
    
    Args:
        start_dt: 起始日, 非 None 表示截取 start_dt 之后的时点
        end_dt: 结束日, 非 None 表示截取 end_dt 之前的时点
    
    Returns:
        时点序列, 若为空 list, 表示该风险表没有固定的时点序列或者无法获取


In [12]:
# 获取风险表的时点序列
DTs = RT.getDateTime()
print(DTs[0], " - ", DTs[-1])

2025-01-01 00:00:00  -  2025-01-10 00:00:00


In [13]:
# 给定起始时点和截止时点, 获取风险表的时点序列
RT.getDateTime(start_dt=dt.datetime(2025, 1, 5), end_dt=dt.datetime(2025, 1, 8))

[datetime.datetime(2025, 1, 5, 0, 0),
 datetime.datetime(2025, 1, 6, 0, 0),
 datetime.datetime(2025, 1, 7, 0, 0),
 datetime.datetime(2025, 1, 8, 0, 0)]

## ID 序列

In [14]:
# 获取风险表 ID 序列的方法
print(qs_help(RT.getID))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.getID(idt: Optional[datetime.datetime] = None) -> List[str]
文档:
    获取 ID 序列
    
    Args:
        idt: 给定的时点, 非 None 表示获取该时点的 ID 序列, None 表示获取所有的 ID 序列
    
    Returns:
        ID 序列, 若为空 list, 表示该风险表没有固定的 ID 序列或者无法获取


In [15]:
# 获取风险表的 ID 序列
IDs = RT.getID()
print(IDs[0], ", ..., ", IDs[-1])

000001.SZ , ...,  000020.SZ


In [16]:
# 给定目标时点, 获取风险表中指定时点的 ID 序列
IDs = RT.getID(idt=dt.datetime(2025, 1, 5))
print(IDs[0], ", ..., ", IDs[-1])

000001.SZ , ...,  000020.SZ


## 读取数据

In [17]:
# 风险表数据读取方法
print(qs_help(RT.readCov))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.RiskTable
签名: FactorRT.readCov(dts: List[datetime.datetime], ids: Optional[List[str]] = None) -> QuantStudio.Core.QSObject.Panel
文档:
    读取风险矩阵
    
    Args:
        dts: 时点序列
        ids: ID 序列, None 表示读取所有的 ID
    
    Returns:
        Panel(item=dts, major_axis=ids, minor_axis=ids)


In [19]:
# 给定时点列表, ID 列表, 获取风险数据
DTs = RT.getDateTime()
Data = RT.readCov(dts=DTs, ids=["000001.SZ", "000002.SZ", "000003.SZ", "000990.SZ"])
print("三维数据 : ")
print(Data)

iDT = dt.datetime(2025, 1, 5)
print(f"时点切片 : {iDT}")
print(Data.loc[iDT])

iID = "000001.SZ"
print(f"ID 切片 : {iID}")
print(Data.loc[:, iID, iID])

三维数据 : 
<class 'QuantStudio.Tools.QSObjects.Panel'>
Dimensions: 10 (items) x 4 (major_axis) x 4 (minor_axis)
Items axis: 2025-01-01 00:00:00 to 2025-01-10 00:00:00
Major_axis axis: 000001.SZ to 000990.SZ
Minor_axis axis: 000001.SZ to 000990.SZ
时点切片 : 2025-01-05 00:00:00
           000001.SZ  000002.SZ  000003.SZ  000990.SZ
000001.SZ   4.227104  -2.186911  -1.738707        NaN
000002.SZ  -2.186911   5.908016  -1.526426        NaN
000003.SZ  -1.738707  -1.526426   5.851773        NaN
000990.SZ        NaN        NaN        NaN        NaN
ID 切片 : 000001.SZ
2025-01-01     9.244580
2025-01-02    10.303747
2025-01-03    11.730036
2025-01-04     8.053961
2025-01-05     4.227104
2025-01-06    17.100300
2025-01-07     8.680541
2025-01-08    11.932926
2025-01-09     8.410562
2025-01-10    16.003128
dtype: float64


# 多因子风险表

## 因子列表

In [20]:
# 获取风险表中的所有因子列表
print(RT.FactorNames)

['Beta', 'BookToPrice', 'EarningsYield', 'Growth', 'Leverage', 'Liquidity', 'Momentum', 'NonlinearSize', 'ResidualVolatility', 'Size']


## 时点序列

### 因子收益时点序列

In [ ]:
# 风险表因子收益时点序列读取方法
print(qs_help(RT.getFactorReturnDateTime))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.getFactorReturnDateTime(start_dt: Optional[datetime.datetime] = None, end_dt: Optional[datetime.datetime] = None) -> List[datetime.datetime]
文档:
    获取因子收益的时点序列
    
    Args:
        start_dt: 起始日, 非 None 表示截取 start_dt 之后的时点
        end_dt: 结束日, 非 None 表示截取 end_dt 之前的时点
    
    Returns:
        时点序列, 若为空 list, 表示该风险表没有固定的因子收益时点序列或者无法获取


In [22]:
# 获取风险表因子收益的时点序列
DTs = RT.getFactorReturnDateTime(dt.datetime(2025, 1, 5), end_dt=dt.datetime(2025, 1, 8))
print(DTs[0], " - ", DTs[-1])

2025-01-05 00:00:00  -  2025-01-08 00:00:00


### 特异性收益时点序列

getSpecificReturnDateTime(start_dt=None, end_dt=None):
* start_dt: datetime 或者 None, 起始时点
* end_dt: datetime 或者 None, 终止时点
* 返回: list(datetime)

In [23]:
# 风险表特异性收益时点序列读取方法
print(qs_help(RT.getSpecificReturnDateTime))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.getSpecificReturnDateTime(start_dt: Optional[datetime.datetime] = None, end_dt: Optional[datetime.datetime] = None) -> List[datetime.datetime]
文档:
    获取特异性收益的时点序列
    
    Args:
        start_dt: 起始日, 非 None 表示截取 start_dt 之后的时点
        end_dt: 结束日, 非 None 表示截取 end_dt 之前的时点
    
    Returns:
        时点序列, 若为空 list, 表示该风险表没有固定的特异性收益时点序列或者无法获取


In [24]:
# 获取风险表特异性收益的时点序列
DTs = RT.getSpecificReturnDateTime(dt.datetime(2025, 1, 5), end_dt=dt.datetime(2025, 1, 8))
print(DTs[0], " - ", DTs[-1])

2025-01-05 00:00:00  -  2025-01-08 00:00:00


## 读取数据

### 因子协方差阵

In [25]:
# 因子协方差矩阵读取方法
print(qs_help(RT.readFactorCov))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.readFactorCov(dts: List[datetime.datetime]) -> QuantStudio.Core.QSObject.Panel
文档:
    读取因子风险矩阵
    
    Args:
        dts: 时点序列
    
    Returns:
        Panel(item=dts, major_axis=[因子], minor_axis=[因子])


In [32]:
# 给定时点列表, 获取因子风险数据
DTs = RT.getDateTime()
Data = RT.readFactorCov(dts=DTs)
print("三维数据 : ")
print(Data)

iDT = dt.datetime(2025, 1, 5)
print(f"时点切片 : {iDT}")
print(Data.loc[iDT].iloc[:3, :3])

三维数据 : 
<class 'QuantStudio.Tools.QSObjects.Panel'>
Dimensions: 10 (items) x 10 (major_axis) x 10 (minor_axis)
Items axis: 2025-01-01 00:00:00 to 2025-01-10 00:00:00
Major_axis axis: Size to Leverage
Minor_axis axis: Size to Leverage
时点切片 : 2025-01-05 00:00:00
              Size      Beta  Momentum
Size      1.417549 -0.145451 -0.030977
Beta     -0.145451  1.026905 -0.077988
Momentum -0.030977 -0.077988  0.987662


### 特异性风险

In [29]:
# 特异性风险读取方法
print(qs_help(RT.readSpecificRisk))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.readSpecificRisk(dts: List[datetime.datetime], ids: Optional[List[str]] = None) -> pandas.core.frame.DataFrame
文档:
    读取特异性风险
    
    Args:
        dts: 时点序列
        ids: 证券 ID 序列, None 表示获取表里所有的 ID
    
    Returns:
        DataFrame(index=dts, columns=ids)


In [31]:
# 读取特异性风险
DTs = RT.getDateTime()
Data = RT.readSpecificRisk(dts=DTs)
print(Data.iloc[:5, :3])

            000001.SZ  000002.SZ  000003.SZ
2025-01-01   0.656127   0.849046   0.439883
2025-01-02   0.753743   0.881754   0.627403
2025-01-03   0.266424   0.297499   0.108461
2025-01-04   0.062444   0.326471   0.967311
2025-01-05   0.111572   0.110870   0.275894


### 因子数据

In [33]:
# 因子暴露数据读取方法
print(qs_help(RT.readFactorData))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.readFactorData(dts: List[datetime.datetime], ids: Optional[List[str]] = None) -> QuantStudio.Core.QSObject.Panel
文档:
    读取因子截面数据
    
    Args:
        dts: 时点序列
        ids: 证券 ID 序列, None 表示获取表里所有的 ID
    
    Returns:
        Panel(items=[因子], major_axis=dts, minor_axis=ids)


In [34]:
# 读取截面因子数据
DTs = RT.getFactorReturnDateTime()
Data = RT.readFactorData(dts=DTs)
print("三维数据 : ")
print(Data)

iDT = dt.datetime(2025, 1, 5)
print(f"时点切片 : {iDT}")
print(Data.loc[:, iDT].iloc[:5, :3])

三维数据 : 
<class 'QuantStudio.Tools.QSObjects.Panel'>
Dimensions: 10 (items) x 10 (major_axis) x 20 (minor_axis)
Items axis: Size to Leverage
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-10 00:00:00
Minor_axis axis: 000001.SZ to 000020.SZ
时点切片 : 2025-01-05 00:00:00
               Size      Beta  Momentum
000001.SZ -1.165150 -0.110541  0.771406
000002.SZ  0.900826  1.020173  1.029439
000003.SZ  0.465662 -0.692050 -0.908763
000004.SZ -1.536244  1.536377 -0.424318
000005.SZ  1.488252  0.286344  0.862596


### 因子收益率

In [35]:
# 因子收益率数据读取方法
print(qs_help(RT.readFactorReturn))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.readFactorReturn(dts: List[datetime.datetime]) -> pandas.core.frame.DataFrame
文档:
    读取因子收益率
    
    Args:
        dts: 时点序列
    
    Returns:
        DataFrame(index=dts, columns=[因子])


In [36]:
# 读取因子收益率
DTs = RT.getFactorReturnDateTime()
Data = RT.readFactorReturn(dts=DTs)
print(Data.iloc[:5, :3])

                Size      Beta  Momentum
2025-01-01 -2.603613  0.231241  0.062179
2025-01-02  0.011558 -0.933668  1.946205
2025-01-03  0.616464  0.614043  0.759888
2025-01-04 -2.292220 -1.493362 -0.237007
2025-01-05  1.187965 -0.693717  0.083646


### 特异性收益率

In [37]:
# 特异性收益率数据读取方法
print(qs_help(RT.readSpecificReturn))

类型: method (bound to HDF5FactorRiskTable)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FactorRiskTable.readSpecificReturn(dts: List[datetime.datetime], ids: Optional[List[str]] = None) -> pandas.core.frame.DataFrame
文档:
    读取特异性收益率
    
    Args:
        dts: 时点序列
        ids: 证券 ID 序列, None 表示获取表里所有的 ID
    
    Returns:
        DataFrame(index=dts, columns=ids)


In [38]:
# 读取特异性收益率
DTs = RT.getSpecificReturnDateTime()
Data = RT.readSpecificReturn(dts=DTs)
print(Data.iloc[:5, :3])

            000001.SZ  000002.SZ  000003.SZ
2025-01-01  -0.759932  -1.740602  -0.356938
2025-01-02   1.254037  -0.127511   0.133909
2025-01-03  -0.090371  -0.703806  -1.861818
2025-01-04  -0.230125   1.051879   0.221146
2025-01-05   0.786102  -0.265291  -0.888391


# 数据写入

## 风险库

In [ ]:
# 数据写入方法
from QuantStudio.Risk.HDF5RDB import HDF5RDB

print(qs_help(HDF5RDB.writeData))

类型: function
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5RDB.writeData(self, table_name: str, idt: datetime.datetime, icov: pandas.core.frame.DataFrame, **kwargs)
文档:
    写入风险数据
    
    Args:
        table_name: 风险表名称
        idt: 待写入的时点
        icov: 风险数据


## 多因子风险库

In [39]:
# 数据写入方法
print(qs_help(RDB.writeData))

类型: method (bound to HDF5FRDB)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FRDB.writeData(table_name: str, idt: datetime.datetime, factor_data: Optional[pandas.core.frame.DataFrame] = None, factor_cov: Optional[pandas.core.frame.DataFrame] = None, specific_risk: Optional[pandas.core.series.Series] = None, factor_ret: Optional[pandas.core.series.Series] = None, specific_ret: Optional[pandas.core.series.Series] = None, **kwargs)
文档:
    写入风险数据
    
    Args:
        table_name: 风险表名称
        idt: 待写入的时点
        factor_data: 因子截面数据, DataFrame(index=[ID], columns=[因子]), 其中 index 是证券代码, columns 是因子列表
        factor_cov: 因子协方差矩阵, DataFrame(index=[因子], columns=[因子]), 其中 index 和 columns 都是因子列表
        specific_risk: 特异性风险: Series(index=[ID]), 其中 index 是证券代码
        factor_ret: 因子收益率, Series(index=[因子]), 其中 index 是因子列表
        specific_ret: 特异性收益率, Series(index=[ID]), 其中 index 是证券代码


In [43]:
# 数据写入
RT = RDB.getTable("demo_risk_table")
CovDTs = RT.getDateTime()[-2:]
IDs = RT.getID()[:5]
FactorCov = RT.readFactorCov(dts=CovDTs)
print("待写入的因子风险数据 : ")
print(FactorCov.iloc[:4, :4])
SpecificRisk = RT.readSpecificRisk(dts=CovDTs, ids=IDs)
print("待写入的特异性风险数据 : ")
print(SpecificRisk)
DTs = RT.getFactorReturnDateTime(start_dt=CovDTs[0]-dt.timedelta(31), end_dt=CovDTs[-1])[:10]
FactorData = RT.readFactorData(dts=DTs, ids=IDs)
print("待写入的因子数据 : ")
print(FactorData)
FactorReturn = RT.readFactorReturn(dts=DTs)
print("待写入的因子收益数据 : ")
print(FactorReturn.iloc[:, :4])
SpecificReturn = RT.readSpecificReturn(dts=DTs, ids=IDs)
print("待写入的特异性收益数据 : ")
print(SpecificReturn)
for iDT in DTs:
    if iDT in CovDTs:
        RDB.writeData(table_name="TestTable", idt=iDT, 
                      factor_data=FactorData.loc[:, iDT], factor_cov=FactorCov.loc[iDT], 
                      specific_risk=SpecificRisk.loc[iDT], factor_ret=FactorReturn.loc[iDT], 
                      specific_ret=SpecificReturn.loc[iDT])
    else:
        RDB.writeData(table_name="TestTable", idt=iDT, 
                      factor_data=FactorData.loc[:, iDT], factor_cov=None, 
                      specific_risk=None, factor_ret=FactorReturn.loc[iDT], 
                      specific_ret=SpecificReturn.loc[iDT])

print("写入后的风险表 : ")
print(RDB.TableNames)

待写入的因子风险数据 : 
<class 'QuantStudio.Tools.QSObjects.Panel'>
Dimensions: 2 (items) x 4 (major_axis) x 10 (minor_axis)
Items axis: 2025-01-09 00:00:00 to 2025-01-10 00:00:00
Major_axis axis: Size to ResidualVolatility
Minor_axis axis: Size to Leverage
待写入的特异性风险数据 : 
            000001.SZ  000002.SZ  000003.SZ  000004.SZ  000005.SZ
2025-01-09   0.429026   0.314148   0.650281   0.884703   0.566562
2025-01-10   0.798174   0.260500   0.804034   0.131800   0.775178
待写入的因子数据 : 
<class 'QuantStudio.Tools.QSObjects.Panel'>
Dimensions: 10 (items) x 10 (major_axis) x 5 (minor_axis)
Items axis: Size to Leverage
Major_axis axis: 2025-01-01 00:00:00 to 2025-01-10 00:00:00
Minor_axis axis: 000001.SZ to 000005.SZ
待写入的因子收益数据 : 
                Size      Beta  Momentum  ResidualVolatility
2025-01-01 -2.603613  0.231241  0.062179            1.317423
2025-01-02  0.011558 -0.933668  1.946205           -2.126034
2025-01-03  0.616464  0.614043  0.759888            1.304775
2025-01-04 -2.292220 -1.493362 -0.2370

# 风险库其他操作

## 设置表的元信息

In [ ]:
# 设置表元信息方法
print(qs_help(RDB.setTableMetaData))

类型: method (bound to HDF5FRDB)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FRDB.setTableMetaData(table_name: str, key: Optional[str] = None, value: Any = None, meta_data: Optional[dict] = None)
文档:
    设置风险表的元信息, 元信息由若干个键值对组成
    
    Args:
        table_name: 风险表名称
        key: 元信息键
        value: 元信息值
        meta_data: 若干组键值对元信息


In [45]:
# 设置表的元信息
TargetTable = "TestTable"
RT = RDB.getTable(TargetTable)
print("设置前的元信息 : ")
print(RT.getMetaData())
RDB.setTableMetaData(table_name=TargetTable, key="Description", value="这是一张测试表")
print("设置后的元信息 : ")
print(RT.getMetaData())

设置前的元信息 : 
Series([], dtype: object)
设置后的元信息 : 
Description    这是一张测试表
dtype: object


## 重命名表

In [46]:
# 重命名表方法
print(qs_help(RDB.renameTable))

类型: method (bound to HDF5FRDB)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FRDB.renameTable(old_table_name: str, new_table_name: str)
文档:
    重命名表
    
    Args:
        old_table_name: 原表名
        new_table_name: 新表名


In [47]:
# 重命名表
print("重命名前风险表 : ")
print(RDB.TableNames)
RDB.renameTable(old_table_name="TestTable", new_table_name="TestTable_New")
print("重命名后风险表 : ")
print(RDB.TableNames)

重命名前风险表 : 
['TestTable', 'demo_risk_table']
重命名后风险表 : 
['TestTable_New', 'demo_risk_table']


## 删除表

In [48]:
# 删除表方法
print(qs_help(RDB.deleteTable))

类型: method (bound to HDF5FRDB)
模块: QuantStudio.Risk.HDF5RDB
签名: HDF5FRDB.deleteTable(table_name: str)
文档:
    删除表
    
    Args:
        table_name: 表名


In [49]:
# 删除表
print("删除前风险表 : ")
print(RDB.TableNames)
RDB.deleteTable(table_name="TestTable_New")
print("删除后因子表 : ")
print(RDB.TableNames)

删除前风险表 : 
['TestTable_New', 'demo_risk_table']
删除后因子表 : 
['demo_risk_table']
